# 04 · Variance reduction: CUPED and regression adjustment

**Goal:** use what was known about customers *before* the emails were sent to make the effect estimates more precise.

Randomization already makes the simple differences in notebook 03 **unbiased**. Variance reduction does not fix bias; it removes noise. Narrower confidence intervals mean the same precision with fewer customers, or the same customers detecting smaller effects (notebook 05 turns this into sample sizes).

Two methods, from simplest to broader:

1. **CUPED** with past-year spend (`history`) as the pre-experiment covariate.
2. **Regression adjustment** with all pre-treatment covariates.

In [1]:
import numpy as np
import pandas as pd

from hillstrom.data import load_customers
from hillstrom.effects import diff_means_welch
from hillstrom.plots import ARM_LABELS
from hillstrom.variance import cuped_adjust, regression_adjusted_effect

df = load_customers()
EMAILS = ["mens", "womens"]
METRICS = ["visit", "conversion", "spend"]

## 1. CUPED

CUPED (Controlled-experiment Using Pre-Experiment Data) replaces each outcome $Y$ by

$$Y^{\text{adj}} = Y - \theta\,(X - \bar X), \qquad \theta = \frac{\operatorname{cov}(Y, X)}{\operatorname{var}(X)},$$

where $X$ is measured before the experiment. Because the emails cannot change $X$, $\bar X$ is the same in expectation in every arm, so the difference in mean $Y^{\text{adj}}$ between arms estimates the same effect as before. Its variance is smaller by the factor $1 - \rho^2$, where $\rho$ is the correlation between $X$ and $Y$. $\theta$ is estimated on both arms pooled.

How much CUPED can help therefore depends entirely on $\rho$. The correlations between the pre-treatment variables and the two-week outcomes:

In [2]:
df[["history", "recency", "visit", "conversion", "spend"]].corr().loc[["history", "recency"], METRICS].style.format("{:+.3f}")

,visit,conversion,spend
history,+0.065,+0.029,+0.022
recency,-0.075,-0.024,-0.016


They are weak: $\rho = 0.065$ between `history` and `visit` means at most $\rho^2 \approx 0.4\%$ variance reduction, and less for conversion and spend. Past-year spend says little about whether a customer visits or buys in a particular two-week window, and the outcomes are mostly zeros.

In [3]:
cuped_rows = []
for email in EMAILS:
    sub = df[df["arm"].isin([email, "control"])]
    treated = (sub["arm"] == email).to_numpy()
    for metric in METRICS:
        y = sub[metric].to_numpy()
        y_adj, theta = cuped_adjust(y, sub["history"].to_numpy())
        raw = diff_means_welch(y[treated], y[~treated])
        adj = diff_means_welch(y_adj[treated], y_adj[~treated])
        cuped_rows.append(
            {
                "contrast": f"{ARM_LABELS[email]} − No E-Mail",
                "metric": metric,
                "theta": theta,
                "raw_estimate": raw.estimate,
                "cuped_estimate": adj.estimate,
                "raw_ci_width": raw.ci_high - raw.ci_low,
                "cuped_ci_width": adj.ci_high - adj.ci_low,
                "variance_reduction": 1 - np.var(y_adj, ddof=1) / np.var(y, ddof=1),
            }
        )
cuped = pd.DataFrame(cuped_rows)
cuped.style.format(
    {"theta": "{:.2e}", "raw_estimate": "{:+.4f}", "cuped_estimate": "{:+.4f}", "raw_ci_width": "{:.4f}",
     "cuped_ci_width": "{:.4f}", "variance_reduction": "{:.2%}"}
).hide(axis="index")

contrast,metric,theta,raw_estimate,cuped_estimate,raw_ci_width,cuped_ci_width,variance_reduction
Mens E-Mail − No E-Mail,visit,9.13e-05,+0.0766,+0.0764,0.0133,0.0132,0.44%
Mens E-Mail − No E-Mail,conversion,1.10e-05,+0.0068,+0.0068,0.0036,0.0036,0.09%
Mens E-Mail − No E-Mail,spend,1.22e-03,+0.7698,+0.7674,0.5694,0.5693,0.04%
Womens E-Mail − No E-Mail,visit,8.35e-05,+0.0452,+0.0451,0.0127,0.0127,0.40%
Womens E-Mail − No E-Mail,conversion,8.72e-06,+0.0031,+0.0031,0.0032,0.0032,0.07%
Womens E-Mail − No E-Mail,spend,1.03e-03,+0.4244,+0.4227,0.5109,0.5108,0.04%


**Result:** CUPED with past-year spend reduces variance by less than half a percent for visits and by hundredths of a percent for spend. The estimates barely move (as they should, since randomization already balanced `history`), and the intervals are essentially unchanged.

## 2. Regression adjustment with all pre-treatment covariates

CUPED with one covariate is a special case of a regression of the outcome on the treatment indicator and pre-treatment covariates:

$$Y = \alpha + \tau\,\text{treated} + \beta^\top X + \varepsilon.$$

The coefficient $\tau$ is the adjusted effect. Here $X$ contains every pre-treatment variable: `recency`, `history`, `mens`, `womens`, `newbie`, `zip_code`, `channel`. Standard errors are heteroskedasticity-robust (HC1), because the outcomes are binary or zero-heavy and their variance differs across customers.

An alternative is Lin's (2013) estimator, which also interacts the treatment with every covariate. It protects against losing precision when the arms are very unequal in size; with three equal arms its advantage here would be negligible, so the simpler additive model is used.

The variance reduction is measured as $1 - (\text{adjusted CI width} / \text{unadjusted CI width})^2$, both from the same regression framework (the unadjusted one has no covariates). Since the required sample size for a given precision is proportional to the variance, this is also the share of customers an experiment could save.

In [4]:
COVARIATES = ["recency", "history", "mens", "womens", "newbie", "zip_code", "channel"]

reg_rows = []
for email in EMAILS:
    for metric in METRICS:
        raw = regression_adjusted_effect(df, metric, email, covariates=[])
        adj = regression_adjusted_effect(df, metric, email, covariates=COVARIATES)
        raw_w, adj_w = raw.ci_high - raw.ci_low, adj.ci_high - adj.ci_low
        reg_rows.append(
            {
                "contrast": f"{ARM_LABELS[email]} − No E-Mail",
                "metric": metric,
                "raw_estimate": raw.estimate,
                "adjusted_estimate": adj.estimate,
                "adjusted_ci": f"[{adj.ci_low:+.4f}, {adj.ci_high:+.4f}]",
                "variance_reduction": 1 - (adj_w / raw_w) ** 2,
            }
        )
regression = pd.DataFrame(reg_rows)
regression.style.format(
    {"raw_estimate": "{:+.4f}", "adjusted_estimate": "{:+.4f}", "variance_reduction": "{:.2%}"}
).hide(axis="index")

contrast,metric,raw_estimate,adjusted_estimate,adjusted_ci,variance_reduction
Mens E-Mail − No E-Mail,visit,+0.0766,+0.0761,"[+0.0695, +0.0826]",2.88%
Mens E-Mail − No E-Mail,conversion,+0.0068,+0.0068,"[+0.0050, +0.0086]",0.28%
Mens E-Mail − No E-Mail,spend,+0.7698,+0.7669,"[+0.4826, +1.0512]",0.25%
Womens E-Mail − No E-Mail,visit,+0.0452,+0.0454,"[+0.0392, +0.0517]",2.64%
Womens E-Mail − No E-Mail,conversion,+0.0031,+0.0031,"[+0.0015, +0.0047]",0.20%
Womens E-Mail − No E-Mail,spend,+0.4244,+0.4246,"[+0.1694, +0.6799]",0.17%


**Result:** using all covariates does better than CUPED with `history` alone, but the gain is still small: about 3% variance reduction for visits and well under 1% for conversion and spend. The adjusted estimates stay within a fraction of a standard error of the unadjusted ones, another sign that randomization worked.

## Takeaways

- **Variance reduction is small in this dataset.** At best about 3% for visits; essentially none for spend. In sample-size terms, a repeat of this experiment could use about 3% fewer customers for the same precision on visits, and practically no fewer for spend.
- **Why:** CUPED and regression adjustment remove the part of the outcome that the covariates predict. Here the covariates are last year's spend and customer attributes, while the outcomes are visits and purchases in one specific two-week window, mostly zeros. The correlation is weak, so there is little to remove.
- **When it would work:** CUPED is most effective when the covariate is the *same metric* measured before the experiment, for example each customer's visits or spend in the two weeks before the email. Hillstrom's data does not include that. For a team running such experiments, collecting the pre-period version of each key metric is the cheapest way to make every future test more sensitive.
- **The main effects in notebook 03 stand unchanged**, with or without adjustment.

**Link to my MSc thesis.** Covariate adjustment is also the theme of my MSc thesis, which studied ROC curve length when covariates are taken into account. The shared idea is to separate the variation that known covariates explain from the variation that matters for the question, here the treatment effect. The methods differ; the principle is the same.

**What we cannot claim:** that covariate adjustment is not worth doing in general. Its value depends on how strongly the pre-experiment data predicts the outcome, and in this dataset the available covariates simply do not predict two-week behaviour well.